In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report)
import joblib
import warnings
warnings.filterwarnings('ignore')

#Load data
df = pd.read_csv('../data/processed/featured_churn_data.csv')

print('---- Preparing Data ----')

#Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Churn ratio: {y.mean()*100:.1f}%")

#Train/test split (stratified to maintain class distribution 27% churn, 73% no churn)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Train churn ratio: {y_train.mean()*100:.1f}%")
print(f"Test churn ratio: {y_test.mean()*100:.1f}%")

#Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Save scaler and feature names
joblib.dump(scaler, '../data/models/scaler.pkl')
joblib.dump(X.columns.tolist(), '../data/models/feature_names.pkl')

print("\n Data prepared and scaler saved")

print("\n---- Training models ----")

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    
    #Train
    model.fit(X_train_scaled, y_train)

    #Predictions
    y_pred_train = model.predict(X_train_scaled)
    y_pred_test = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

    #Metrics
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    precision = precision_score(y_test, y_pred_test)
    recall = recall_score(y_test, y_pred_test)
    f1 = f1_score(y_test, y_pred_test)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    results[name] = {
        'model': model,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc,
        'y_pred': y_pred_test,
        'y_pred_proba': y_pred_proba
    }

    print(f" Train acc: {train_acc:.3f} | Test acc: {test_acc:.3f}")
    print(f" Presicion: {precision:.3f} | Recall: {recall:.3f}")
    print(f" F1: {f1:.3f} | ROC-AUC: {roc_auc:.3f}\n")

#Compare models
print("---- Model Comparison ----")
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Test Accuracy': [results[m]['test_acc'] for m in results.keys()],
    'Precision': [results[m]['precision'] for m in results.keys()],
    'Recall': [results[m]['recall'] for m in results.keys()],
    'F1-Score': [results[m]['f1'] for m in results.keys()],
    'ROC-AUC': [results[m]['roc_auc'] for m in results.keys()]
})

print(comparison_df.to_string(index=False))

#Select best model
best_model_name = comparison_df.loc[comparison_df['ROC-AUC'].idxmax(), 'Model']
best_model = results[best_model_name]['model']

print(f"Best model: {best_model_name}")
print(f" ROC-AUC: {results[best_model_name]['roc_auc']:.3f}")

#Save best model
joblib.dump(best_model, '../data/models/best_model.pkl')
print("\n Best model saved")

---- Preparing Data ----
Features shape: (7043, 28)
Target shape: (7043,)
Churn ratio: 26.5%

Train set: 5634 samples
Test set: 1409 samples
Train churn ratio: 26.5%
Test churn ratio: 26.5%

 Data prepared and scaler saved

---- Training models ----
Training Logistic Regression...
 Train acc: 0.807 | Test acc: 0.805
 Presicion: 0.667 | Recall: 0.529
 F1: 0.590 | ROC-AUC: 0.845

Training Random Forest...
 Train acc: 0.992 | Test acc: 0.798
 Presicion: 0.653 | Recall: 0.508
 F1: 0.571 | ROC-AUC: 0.835

Training Gradient Boosting...
 Train acc: 0.873 | Test acc: 0.798
 Presicion: 0.644 | Recall: 0.532
 F1: 0.583 | ROC-AUC: 0.836

---- Model Comparison ----
              Model  Test Accuracy  Precision   Recall  F1-Score  ROC-AUC
Logistic Regression       0.804826   0.666667 0.529412  0.590164 0.845199
      Random Forest       0.797729   0.652921 0.508021  0.571429 0.834984
  Gradient Boosting       0.797729   0.644013 0.532086  0.582723 0.835895
Best model: Logistic Regression
 ROC-AUC: 